# 3. Segmentacja Obiektów Pierwszoplanowych

In [1]:
import os
import cv2
import glob
import numpy as np


In [16]:
def segment_sequence(dataset_path, threshold_value, buffer_size=60, median=False):
	input_path = os.path.join(dataset_path, 'input', "*.jpg")
	gt_path = os.path.join(dataset_path, 'groundtruth', '*.png')
	
	input_files = sorted(glob.glob(input_path))
	gt_files = sorted(glob.glob(gt_path))

	if not input_files or not gt_files:
		print("Nie znaleziono plików w katalogu.")
		return None, None, None
	
	roi_file = os.path.join(dataset_path, "temporalROI.txt")
	if os.path.exists(roi_file):
		with open(roi_file, "r") as f:
			start_frame, end_frame = map(int, f.read().split())
	else:
		start_frame, end_frame = 1, len(input_files)
	print(f"Analizowane klatki to {start_frame} - {end_frame}")

	first_frame = cv2.imread(input_files[0], cv2.IMREAD_GRAYSCALE)

	# Inicjalizacja bufora i licznika
	buffer = np.zeros((first_frame.shape[0], first_frame.shape[1], buffer_size), np.uint8)

	# wypelnienie calego bufora pierwsza klatka
	for i in range(buffer_size):
		buffer[:, :, i] = first_frame
	cnt = 0
		
	total_TP, total_FP, total_FN = 0, 0, 0
	
	# glowna petla
	for i in range(start_frame - 1, end_frame):
		gt_mask = cv2.imread(gt_files[i], cv2.IMREAD_GRAYSCALE)
		_, binary_gt = cv2.threshold(gt_mask, 254, 255, cv2.THRESH_BINARY)

		curr_frame = cv2.imread(input_files[i], cv2.IMREAD_GRAYSCALE)

		# obliczanie background z poprzedniego stanu modelu
		if median:
			bg_model = np.median(buffer, axis=2).astype(np.uint8)
		else:
			bg_model = np.mean(buffer, axis=2).astype(np.uint8)
		
		diff = np.abs(curr_frame.astype(np.int16) - bg_model.astype(np.int16)).astype(np.uint8)
		_, binary_diff = cv2.threshold(diff, threshold_value, 255, cv2.THRESH_BINARY)

		blur_diff = cv2.medianBlur(binary_diff, 5)
		morph_diff = cv2.morphologyEx(blur_diff, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8), iterations=2)

		TP = np.sum(np.logical_and(morph_diff == 255, binary_gt == 255))
		FP = np.sum(np.logical_and(morph_diff == 255, binary_gt == 0))
		FN = np.sum(np.logical_and(morph_diff == 0, binary_gt == 255))

		total_TP += TP
		total_FP += FP
		total_FN += FN

		cv2.imshow("Oryginal", curr_frame)
		cv2.imshow("Surowa roznica (wynik - maska)", binary_diff)
		cv2.imshow("Po operacjach morfologicznych", morph_diff)

		if cv2.waitKey(10) & 0xFF == 27:
			print("Przerwano przez uzytkownika")
			break

		# aktualizacja modelu tla po detekcji dla biezacej klatki
		buffer[:, :, cnt] = curr_frame
		cnt = (cnt + 1) % buffer_size

	cv2.destroyAllWindows()

	print("=== PODSUMOWANIE WYNIKOW ===")

	precision, recall, f1_score = 0, 0, 0
	if (total_TP + total_FP) > 0:
		precision = total_TP / (total_TP + total_FP)
	
	if (total_TP + total_FN) > 0:
		recall = total_TP / (total_TP + total_FN)

	if (precision + recall) > 0:
		f1_score = 2 * (precision * recall) / (precision + recall)

	print(f"Precision: {precision:.4f}")
	print(f"Recall: {recall:.4f}")
	print(f"F1-Score: {f1_score:.4f}")

In [19]:
datasets = ["highway", "office", "pedestrian"]
for ds in datasets:
	segment_sequence(ds, 40)


Analizowane klatki to 470 - 1700
=== PODSUMOWANIE WYNIKOW ===
Precision: 0.6127
Recall: 0.7492
F1-Score: 0.6741
Analizowane klatki to 570 - 2050
=== PODSUMOWANIE WYNIKOW ===
Precision: 0.4815
Recall: 0.2367
F1-Score: 0.3174
Analizowane klatki to 300 - 1099
=== PODSUMOWANIE WYNIKOW ===
Precision: 0.6528
Recall: 0.8892
F1-Score: 0.7529


## Przyspieszenie algorytmu
Każdorazowe obliczenie średniej jest kosztowne pamięciowo i obliczniowo. Jeszcze gorzej to wygląda jeśli chodzi o mediane. Dlatego stosuje się metody, które nie wymagają bufora i aproksymują średnią oraz mediane

In [20]:
def segment_items_approx(dataset_path, threshold_value, median=False, alpha=0.01):
	input_path = os.path.join(dataset_path, 'input', "*.jpg")
	gt_path = os.path.join(dataset_path, 'groundtruth', '*.png')
	
	input_files = sorted(glob.glob(input_path))
	gt_files = sorted(glob.glob(gt_path))

	if not input_files or not gt_files:
		print("Nie znaleziono plikow w katalogu.")
		return None, None, None
	
	roi_file = os.path.join(dataset_path, "temporalROI.txt")
	if os.path.exists(roi_file):
		with open(roi_file, "r") as f:
			start_frame, end_frame = map(int, f.read().split())
	else:
		start_frame, end_frame = 1, len(input_files)
	print(f"Analizowane klatki to {start_frame} - {end_frame}")

	first_frame = cv2.imread(input_files[0], cv2.IMREAD_GRAYSCALE)
	bg_model = first_frame.copy().astype(np.float64)

	total_TP, total_FP, total_FN = 0, 0, 0
	
	# glowna petla
	for i in range(start_frame - 1, end_frame):
		gt_mask = cv2.imread(gt_files[i], cv2.IMREAD_GRAYSCALE)
		_, binary_gt = cv2.threshold(gt_mask, 254, 255, cv2.THRESH_BINARY)

		curr_frame = cv2.imread(input_files[i], cv2.IMREAD_GRAYSCALE)

		# detekcja na modelu tla sprzed aktualizacji
		bg_model_uint = np.clip(bg_model, 0, 255).astype(np.uint8)
		diff = np.abs(curr_frame.astype(np.int16) - bg_model_uint.astype(np.int16)).astype(np.uint8)
		_, binary_diff = cv2.threshold(diff, threshold_value, 255, cv2.THRESH_BINARY)

		blur_diff = cv2.medianBlur(binary_diff, 5)
		morph_diff = cv2.morphologyEx(blur_diff, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8), iterations=3)

		TP = np.sum(np.logical_and(morph_diff == 255, binary_gt == 255))
		FP = np.sum(np.logical_and(morph_diff == 255, binary_gt == 0))
		FN = np.sum(np.logical_and(morph_diff == 0, binary_gt == 255))

		total_TP += TP
		total_FP += FP
		total_FN += FN

		cv2.imshow("Oryginal", curr_frame)
		cv2.imshow("Surowa roznica (wynik - maska)", binary_diff)
		cv2.imshow("Po operacjach morfologicznych", morph_diff)

		if cv2.waitKey(10) & 0xFF == 27:
			print("Przerwano przez uzytkownika")
			break

		# aktualizacja modelu tla po detekcji
		if median:
			bg_model += (bg_model < curr_frame).astype(np.float64)
			bg_model -= (bg_model > curr_frame).astype(np.float64)
		else:
			bg_model = alpha * curr_frame + (1 - alpha) * bg_model

	cv2.destroyAllWindows()

	print("=== PODSUMOWANIE WYNIKOW ===")

	precision, recall, f1_score = 0, 0, 0
	if (total_TP + total_FP) > 0:
		precision = total_TP / (total_TP + total_FP)
	
	if (total_TP + total_FN) > 0:
		recall = total_TP / (total_TP + total_FN)

	if (precision + recall) > 0:
		f1_score = 2 * (precision * recall) / (precision + recall)

	print(f"Precision: {precision:.4f}")
	print(f"Recall: {recall:.4f}")
	print(f"F1-Score: {f1_score:.4f}")

In [22]:
print("\nWYNIKI DLA ŚREDNIEJ")
for ds in datasets:
	segment_items_approx(ds, 40)

print("\nWYNIKI DLA MEDIANY")
for ds in datasets:
	segment_items_approx(ds, 40, median=True)


WYNIKI DLA ŚREDNIEJ
Analizowane klatki to 470 - 1700
=== PODSUMOWANIE WYNIKOW ===
Precision: 0.7250
Recall: 0.9135
F1-Score: 0.8084
Analizowane klatki to 570 - 2050
=== PODSUMOWANIE WYNIKOW ===
Precision: 0.5628
Recall: 0.4390
F1-Score: 0.4932
Analizowane klatki to 300 - 1099
=== PODSUMOWANIE WYNIKOW ===
Precision: 0.6328
Recall: 0.9338
F1-Score: 0.7544

WYNIKI DLA MEDIANY
Analizowane klatki to 470 - 1700
=== PODSUMOWANIE WYNIKOW ===
Precision: 0.7579
Recall: 0.9212
F1-Score: 0.8316
Analizowane klatki to 570 - 2050
=== PODSUMOWANIE WYNIKOW ===
Precision: 0.5770
Recall: 0.3700
F1-Score: 0.4508
Analizowane klatki to 300 - 1099
=== PODSUMOWANIE WYNIKOW ===
Precision: 0.6363
Recall: 0.9323
F1-Score: 0.7564


### Polityka aktualizacji

In [24]:
def segment_items_approx_conservative(dataset_path, threshold_value, median=False, alpha=0.01):
	input_path = os.path.join(dataset_path, 'input', "*.jpg")
	gt_path = os.path.join(dataset_path, 'groundtruth', '*.png')
	
	input_files = sorted(glob.glob(input_path))
	gt_files = sorted(glob.glob(gt_path))

	if not input_files or not gt_files:
		print("Nie znaleziono plikow w katalogu.")
		return None, None, None
	
	roi_file = os.path.join(dataset_path, "temporalROI.txt")
	if os.path.exists(roi_file):
		with open(roi_file, "r") as f:
			start_frame, end_frame = map(int, f.read().split())
	else:
		start_frame, end_frame = 1, len(input_files)
	print(f"Analizowane klatki to {start_frame} - {end_frame}")

	first_frame = cv2.imread(input_files[0], cv2.IMREAD_GRAYSCALE)
	bg_model = first_frame.copy().astype(np.float64)

	# na poczatku zakladamy brak obiektow
	prev_mask = np.zeros_like(first_frame, dtype=np.uint8)
	
	total_TP, total_FP, total_FN = 0, 0, 0
	
	for i in range(start_frame - 1, end_frame):
		gt_mask = cv2.imread(gt_files[i], cv2.IMREAD_GRAYSCALE)
		_, binary_gt = cv2.threshold(gt_mask, 254, 255, cv2.THRESH_BINARY)

		curr_frame = cv2.imread(input_files[i], cv2.IMREAD_GRAYSCALE)
		can_update = (prev_mask == 0)

		# detekcja na modelu tla sprzed aktualizacji
		bg_model_uint = np.clip(bg_model, 0, 255).astype(np.uint8)
		diff = np.abs(curr_frame.astype(np.int16) - bg_model_uint.astype(np.int16)).astype(np.uint8)
		_, binary_diff = cv2.threshold(diff, threshold_value, 255, cv2.THRESH_BINARY)

		blur_diff = cv2.medianBlur(binary_diff, 5)
		morph_diff = cv2.morphologyEx(blur_diff, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8), iterations=2)

		TP = np.sum(np.logical_and(morph_diff == 255, binary_gt == 255))
		FP = np.sum(np.logical_and(morph_diff == 255, binary_gt == 0))
		FN = np.sum(np.logical_and(morph_diff == 0, binary_gt == 255))

		total_TP += TP
		total_FP += FP
		total_FN += FN

		cv2.imshow("Oryginal", curr_frame)
		cv2.imshow("Surowa roznica (wynik - maska)", binary_diff)
		cv2.imshow("Po operacjach morfologicznych", morph_diff)

		if cv2.waitKey(10) & 0xFF == 27:
			print("Przerwano przez uzytkownika")
			break

		# aktualizacja modelu tla po detekcji, tylko dla tla z poprzedniej maski
		if median:
			step_up = (bg_model < curr_frame).astype(np.float64)
			step_down = (bg_model > curr_frame).astype(np.float64)
			bg_model[can_update] -= step_down[can_update]
			bg_model[can_update] += step_up[can_update]
		else:
			bg_model[can_update] = alpha * curr_frame[can_update] + (1 - alpha) * bg_model[can_update]

		prev_mask = morph_diff.copy()

	cv2.destroyAllWindows()

	print("=== PODSUMOWANIE WYNIKOW ===")

	precision, recall, f1_score = 0, 0, 0
	if (total_TP + total_FP) > 0:
		precision = total_TP / (total_TP + total_FP)
	
	if (total_TP + total_FN) > 0:
		recall = total_TP / (total_TP + total_FN)

	if (precision + recall) > 0:
		f1_score = 2 * (precision * recall) / (precision + recall)

	print(f"Precision: {precision:.4f}")
	print(f"Recall: {recall:.4f}")
	print(f"F1-Score: {f1_score:.4f}")

In [25]:
print("\nWYNIKI DLA ŚREDNIEJ")
for ds in datasets:
	segment_items_approx_conservative(ds, 70)

print("\nWYNIKI DLA MEDIANY")
for ds in datasets:
	segment_items_approx_conservative(ds, 70, median=True)


WYNIKI DLA ŚREDNIEJ
Analizowane klatki to 470 - 1700
=== PODSUMOWANIE WYNIKOW ===
Precision: 0.7084
Recall: 0.5253
F1-Score: 0.6033
Analizowane klatki to 570 - 2050
=== PODSUMOWANIE WYNIKOW ===
Precision: 0.8237
Recall: 0.4864
F1-Score: 0.6116
Analizowane klatki to 300 - 1099
=== PODSUMOWANIE WYNIKOW ===
Precision: 0.8048
Recall: 0.7283
F1-Score: 0.7646

WYNIKI DLA MEDIANY
Analizowane klatki to 470 - 1700
=== PODSUMOWANIE WYNIKOW ===
Precision: 0.7098
Recall: 0.5289
F1-Score: 0.6061
Analizowane klatki to 570 - 2050
=== PODSUMOWANIE WYNIKOW ===
Precision: 0.7672
Recall: 0.4749
F1-Score: 0.5867
Analizowane klatki to 300 - 1099
=== PODSUMOWANIE WYNIKOW ===
Precision: 0.8054
Recall: 0.7263
F1-Score: 0.7638


### OpenCV - GMM(Gaussian Mixture Models)/MoG(Mixture of Gaussians)

In [27]:
def segmentGMM(dataset_path):
	input_path = os.path.join(dataset_path, 'input', "*.jpg")
	gt_path = os.path.join(dataset_path, 'groundtruth', '*.png')
	
	input_files = sorted(glob.glob(input_path))
	gt_files = sorted(glob.glob(gt_path))

	if not input_files or not gt_files:
		print("Nie znaleziono plikow w katalogu.")
		return None, None, None
	
	roi_file = os.path.join(dataset_path, "temporalROI.txt")
	if os.path.exists(roi_file):
		with open(roi_file, "r") as f:
			start_frame, end_frame = map(int, f.read().split())
	else:
		start_frame, end_frame = 1, len(input_files)
	print(f"Analizowane klatki to {start_frame} - {end_frame}")

	# history - dlugosc pamieci modelu, mniejsze progi daja wieksza czulosc
	mog2 = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=16, detectShadows=False)
	knn = cv2.createBackgroundSubtractorKNN(history=500, dist2Threshold=400, detectShadows=False)

	total_TP_mog, total_FP_mog, total_FN_mog = 0, 0, 0
	total_TP_knn, total_FP_knn, total_FN_knn = 0, 0, 0

	for i in range(start_frame - 1, end_frame):
		gt_mask = cv2.imread(gt_files[i], cv2.IMREAD_GRAYSCALE)
		_, binary_gt = cv2.threshold(gt_mask, 254, 255, cv2.THRESH_BINARY)
		curr_frame = cv2.imread(input_files[i], cv2.IMREAD_GRAYSCALE)

		binary_diff_mog = mog2.apply(curr_frame, learningRate=-1)
		binary_diff_knn = knn.apply(curr_frame, learningRate=-1)

		blur_diff_mog = cv2.medianBlur(binary_diff_mog, 5)
		blur_diff_knn = cv2.medianBlur(binary_diff_knn, 5)

		morph_diff_mog = cv2.morphologyEx(blur_diff_mog, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8), iterations=2)
		morph_diff_knn = cv2.morphologyEx(blur_diff_knn, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8), iterations=2)

		TP_mog = np.sum(np.logical_and(morph_diff_mog == 255, binary_gt == 255))
		FP_mog = np.sum(np.logical_and(morph_diff_mog == 255, binary_gt == 0))
		FN_mog = np.sum(np.logical_and(morph_diff_mog == 0, binary_gt == 255))

		TP_knn = np.sum(np.logical_and(morph_diff_knn == 255, binary_gt == 255))
		FP_knn = np.sum(np.logical_and(morph_diff_knn == 255, binary_gt == 0))
		FN_knn = np.sum(np.logical_and(morph_diff_knn == 0, binary_gt == 255))

		total_TP_mog += TP_mog
		total_FP_mog += FP_mog
		total_FN_mog += FN_mog

		total_TP_knn += TP_knn
		total_FP_knn += FP_knn
		total_FN_knn += FN_knn

		cv2.imshow("Oryginal", curr_frame)
		cv2.imshow("Wynik z MOG2", binary_diff_mog)
		cv2.imshow("Wynik z KNN", binary_diff_knn)
		cv2.imshow("KNN po operacjach morfologicznych", morph_diff_knn)
		cv2.imshow("MOG2 po operacjach morfologicznych", morph_diff_mog)

		if cv2.waitKey(10) & 0xFF == 27:
			print("Przerwano przez uzytkownika")
			break

	cv2.destroyAllWindows()

	def print_metrics(name, TP, FP, FN):
		precision, recall, f1_score = 0, 0, 0
		if (TP + FP) > 0:
			precision = TP / (TP + FP)
		if (TP + FN) > 0:
			recall = TP / (TP + FN)
		if (precision + recall) > 0:
			f1_score = 2 * (precision * recall) / (precision + recall)
		print(f"=== PODSUMOWANIE WYNIKOW: {name} ===")
		print(f"Precision: {precision:.4f}")
		print(f"Recall: {recall:.4f}")
		print(f"F1-Score: {f1_score:.4f}")

	print_metrics("MOG2", total_TP_mog, total_FP_mog, total_FN_mog)
	print_metrics("KNN", total_TP_knn, total_FP_knn, total_FN_knn)

In [28]:
for ds in datasets:
	segmentGMM(ds)

Analizowane klatki to 470 - 1700
=== PODSUMOWANIE WYNIKOW: MOG2 ===
Precision: 0.7363
Recall: 0.8963
F1-Score: 0.8085
=== PODSUMOWANIE WYNIKOW: KNN ===
Precision: 0.7366
Recall: 0.9380
F1-Score: 0.8252
Analizowane klatki to 570 - 2050
=== PODSUMOWANIE WYNIKOW: MOG2 ===
Precision: 0.5228
Recall: 0.1319
F1-Score: 0.2107
=== PODSUMOWANIE WYNIKOW: KNN ===
Precision: 0.6567
Recall: 0.2355
F1-Score: 0.3467
Analizowane klatki to 300 - 1099
=== PODSUMOWANIE WYNIKOW: MOG2 ===
Precision: 0.4288
Recall: 0.9845
F1-Score: 0.5974
=== PODSUMOWANIE WYNIKOW: KNN ===
Precision: 0.4400
Recall: 0.9670
F1-Score: 0.6048


In [29]:
def segment_objects(dataset_path, threshold_value, buffer_size=60, median_and_mean=True, alpha=0.01):
	"""Funkcja przeprowadza segmentacje obiektow pierwszoplanowych na kilka sposobow:
	1) Metody oparte o bufor probek (wlaczane flaga median_and_mean = True)
		a) algorytm mediany
		b) algorytm sredniej
	2) Aproksymacja sredniej i mediany (sigma-delta)
	3) Konserwatywna polityka aktualizacji
	4) MoG2 oraz KNN"""

	input_path = os.path.join(dataset_path, 'input', "*.jpg")
	gt_path = os.path.join(dataset_path, 'groundtruth', '*.png')
	
	input_files = sorted(glob.glob(input_path))
	gt_files = sorted(glob.glob(gt_path))

	if not input_files or not gt_files:
		print("Nie znaleziono plikow w katalogu.")
		return None
	
	roi_file = os.path.join(dataset_path, "temporalROI.txt")
	if os.path.exists(roi_file):
		with open(roi_file, "r") as f:
			start_frame, end_frame = map(int, f.read().split())
	else:
		start_frame, end_frame = 1, len(input_files)
	print(f"Analizowane klatki to {start_frame} - {end_frame}")

	first_frame = cv2.imread(input_files[0], cv2.IMREAD_GRAYSCALE)
	buffer_mean = np.zeros((first_frame.shape[0], first_frame.shape[1], buffer_size), np.uint8)
	buffer_median = np.zeros((first_frame.shape[0], first_frame.shape[1], buffer_size), np.uint8)
	for i in range(buffer_size):
		buffer_mean[:, :, i] = first_frame
		buffer_median[:, :, i] = first_frame
	cnt = 0

	bg_model_approx_mean = first_frame.copy().astype(np.float64)
	bg_model_approx_median = first_frame.copy().astype(np.float64)

	bg_model_approx_mean_conservative = first_frame.copy().astype(np.float64)
	bg_model_approx_median_conservative = first_frame.copy().astype(np.float64)
	prev_mask_cons_mean = np.zeros_like(first_frame, dtype=np.uint8)
	prev_mask_cons_median = np.zeros_like(first_frame, dtype=np.uint8)

	mog2 = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=16, detectShadows=False)
	knn = cv2.createBackgroundSubtractorKNN(history=500, dist2Threshold=400, detectShadows=False)

	for i in range(start_frame - 1, end_frame):
		curr_frame = cv2.imread(input_files[i], cv2.IMREAD_GRAYSCALE)

		# 1) mediana i srednia z buforem - detekcja przed aktualizacja bufora
		if median_and_mean:
			bg_model_mean = np.mean(buffer_mean, axis=2).astype(np.uint8)
			bg_model_median = np.median(buffer_median, axis=2).astype(np.uint8)
			diff_mean = np.abs(curr_frame.astype(np.int16) - bg_model_mean.astype(np.int16)).astype(np.uint8)
			diff_median = np.abs(curr_frame.astype(np.int16) - bg_model_median.astype(np.int16)).astype(np.uint8)
			_, binary_mean = cv2.threshold(diff_mean, threshold_value, 255, cv2.THRESH_BINARY)
			_, binary_median = cv2.threshold(diff_median, threshold_value, 255, cv2.THRESH_BINARY)
			cv2.imshow("Mediana", binary_median)
			cv2.imshow("Srednia", binary_mean)

		# 2) Aproksymacja sredniej i mediany - detekcja na starym modelu
		bg_approx_mean_uint = np.clip(bg_model_approx_mean, 0, 255).astype(np.uint8)
		bg_approx_median_uint = np.clip(bg_model_approx_median, 0, 255).astype(np.uint8)
		diff_approx_mean = np.abs(curr_frame.astype(np.int16) - bg_approx_mean_uint.astype(np.int16)).astype(np.uint8)
		diff_approx_median = np.abs(curr_frame.astype(np.int16) - bg_approx_median_uint.astype(np.int16)).astype(np.uint8)
		_, binary_approx_mean = cv2.threshold(diff_approx_mean, threshold_value, 255, cv2.THRESH_BINARY)
		_, binary_approx_median = cv2.threshold(diff_approx_median, threshold_value, 255, cv2.THRESH_BINARY)
		cv2.imshow("Aproksymacja Mediany", binary_approx_median)
		cv2.imshow("Aproksymacja Sredniej", binary_approx_mean)

		# 3) Konserwatywna polityka aktualizacji - detekcja na starym modelu
		can_update_mean = (prev_mask_cons_mean == 0)
		can_update_median = (prev_mask_cons_median == 0)
		bg_cons_mean_uint = np.clip(bg_model_approx_mean_conservative, 0, 255).astype(np.uint8)
		bg_cons_median_uint = np.clip(bg_model_approx_median_conservative, 0, 255).astype(np.uint8)
		diff_cons_mean = np.abs(curr_frame.astype(np.int16) - bg_cons_mean_uint.astype(np.int16)).astype(np.uint8)
		diff_cons_median = np.abs(curr_frame.astype(np.int16) - bg_cons_median_uint.astype(np.int16)).astype(np.uint8)
		_, binary_cons_mean = cv2.threshold(diff_cons_mean, threshold_value, 255, cv2.THRESH_BINARY)
		_, binary_cons_median = cv2.threshold(diff_cons_median, threshold_value, 255, cv2.THRESH_BINARY)
		cv2.imshow("Konserwatywna Srednia", binary_cons_mean)
		cv2.imshow("Konserwatywna Mediana", binary_cons_median)

		# 4) MoG2 oraz KNN
		binary_mog = mog2.apply(curr_frame, learningRate=-1)
		binary_knn = knn.apply(curr_frame, learningRate=-1)
		cv2.imshow("MoG2", binary_mog)
		cv2.imshow("KNN", binary_knn)

		cv2.imshow("Oryginalny Obraz", curr_frame)

		if cv2.waitKey(10) & 0xFF == 27:
			print("Przerwano przez uzytkownika")
			break

		# aktualizacje modeli po detekcji
		if median_and_mean:
			buffer_mean[:, :, cnt] = curr_frame
			buffer_median[:, :, cnt] = curr_frame
			cnt = (cnt + 1) % buffer_size

		bg_model_approx_mean = alpha * curr_frame + (1 - alpha) * bg_model_approx_mean
		bg_model_approx_median += (bg_model_approx_median < curr_frame).astype(np.float64)
		bg_model_approx_median -= (bg_model_approx_median > curr_frame).astype(np.float64)

		bg_model_approx_mean_conservative[can_update_mean] = (
			alpha * curr_frame[can_update_mean] + (1 - alpha) * bg_model_approx_mean_conservative[can_update_mean]
		)
		step_up = (bg_model_approx_median_conservative < curr_frame).astype(np.float64)
		step_down = (bg_model_approx_median_conservative > curr_frame).astype(np.float64)
		bg_model_approx_median_conservative[can_update_median] -= step_down[can_update_median]
		bg_model_approx_median_conservative[can_update_median] += step_up[can_update_median]

		prev_mask_cons_mean = binary_cons_mean.copy()
		prev_mask_cons_median = binary_cons_median.copy()

	cv2.destroyAllWindows()

In [30]:
segment_objects("highway", 70, median_and_mean=False)

Analizowane klatki to 470 - 1700
Przerwano przez uzytkownika
